In [4]:



import inspect
import os
import CRUD_Python_Module

from CRUD_Python_Module import AnimalShelter

print("Imported from:")
print(CRUD_Python_Module.__file__)

print("Constructor expects:")
print(inspect.signature(AnimalShelter.__init__))

Imported from:
C:\Users\suare\Downloads\CS-340-main\CS-340-main\CS340Mod7\CRUD_Python_Module.py
Constructor expects:
(self, username, password, host, port, database, collection)


In [5]:
from CRUD_Python_Module import AnimalShelter
import pandas as pd
###########################
# Data Manipulation / Model
###########################

username = os.getenv("AAC_USERNAME")
password = os.getenv("AAC_PASSWORD")

host = os.getenv("AAC_HOST", "localhost")
port = int(os.getenv("AAC_PORT", "27017"))
database = os.getenv("AAC_DATABASE", "aac")
collection = os.getenv("AAC_COLLECTION", "animals")

shelter = AnimalShelter(username, password, host, port, database, collection)

# Test MongoDB connection through the CRUD object
print("MongoDB record count:")
print(shelter.collection.count_documents({}))

print("First record:")
print(shelter.collection.find_one())

# Create the compound indexes used by the dashboard.
print("Database indexes:")
print(shelter.create_indexes())

# Test the breed aggregation pipeline.
print("Top five dog breeds:")
print(
    shelter.aggregate_breed_counts(
        {"animal_type": "Dog"},
        5
    )
)

# Display the newest administrative audit entries.
print("Recent audit entries:")
print(shelter.read_audit_logs(3))

# Load MongoDB data into dataframe
df = pd.DataFrame.from_records(
    shelter.read({})
)

# Remove MongoDB ObjectId column if it exists
df.drop(
    columns=["_id"],
    inplace=True,
    errors="ignore"
)

print("Dataframe rows:", len(df))
print("Columns:")
print(df.columns)

MongoDB record count:
10000
First record:
{'_id': ObjectId('6a4f15a28dfe8e0bdedc1f93'), '': 1, 'age_upon_outcome': '3 years', 'animal_id': 'A746874', 'animal_type': 'Cat', 'breed': 'Domestic Shorthair Mix', 'color': 'Black/White', 'date_of_birth': datetime.datetime(2014, 4, 10, 0, 0), 'datetime': '2017-04-11 09:00:00', 'monthyear': '2017-04-11T09:00:00', 'name': '', 'outcome_subtype': 'SCRP', 'outcome_type': 'Transfer', 'sex_upon_outcome': 'Neutered Male', 'location_lat': 30.5066578739455, 'location_long': -97.3408780722188, 'age_upon_outcome_in_weeks': 156.767857142857}
Dataframe rows: 10000
Columns:
Index(['', 'age_upon_outcome', 'animal_id', 'animal_type', 'breed', 'color',
       'date_of_birth', 'datetime', 'monthyear', 'name', 'outcome_subtype',
       'outcome_type', 'sex_upon_outcome', 'location_lat', 'location_long',
       'age_upon_outcome_in_weeks'],
      dtype='str')


In [1]:
# Setup the Jupyter version of Dash
from dash import Dash

# Configure the necessary Python module imports
import os
import time
import heapq
from functools import lru_cache

import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
#JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


####  #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

username = os.getenv("AAC_USERNAME")
password = os.getenv("AAC_PASSWORD")

host = os.getenv("AAC_HOST", "localhost")
port = int(os.getenv("AAC_PORT", "27017"))
database = os.getenv("AAC_DATABASE", "aac")
collection = os.getenv("AAC_COLLECTION", "animals")

shelter = AnimalShelter(
    username,
    password,
    host,
    port,
    database,
    collection
)


water_query = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "Labrador Retriever Mix",
            "Chesapeake Bay Retriever",
            "Newfoundland"
        ]
    },
    "sex_upon_outcome": "Intact Female",
    "age_upon_outcome_in_weeks": {
        "$gte": 26,
        "$lte": 156
    }
}


wilderness_query = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "German Shepherd",
            "Alaskan",
            "Malamute",
            "Old English Sheepdog",
            "Siberian Husky",
            "Rottweiler"
        ]
    },
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {
        "$gte": 26,
        "$lte": 156
    }
}


disaster_query = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "Doberman Pinscher",
            "German Shepherd",
            "Golden Retriever",
            "Bloodhound",
            "Rottweiler"
        ]
    },
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {
        "$gte": 20,
        "$lte": 300
    }
}


reset_query = {}


#############################################
# Algorithms and Data Structures Enhancement
#############################################

# Number of candidates retained by the heap
TOP_K = 10


# Store the rescue requirements in dictionaries so the same
# scoring algorithm can be used for each rescue category.
rescue_profiles = {
    "water": {
        "breeds": water_query["breed"]["$in"],
        "sex": water_query["sex_upon_outcome"],
        "minimum_age": water_query[
            "age_upon_outcome_in_weeks"
        ]["$gte"],
        "maximum_age": water_query[
            "age_upon_outcome_in_weeks"
        ]["$lte"]
    },

    "wilderness": {
        "breeds": wilderness_query["breed"]["$in"],
        "sex": wilderness_query["sex_upon_outcome"],
        "minimum_age": wilderness_query[
            "age_upon_outcome_in_weeks"
        ]["$gte"],
        "maximum_age": wilderness_query[
            "age_upon_outcome_in_weeks"
        ]["$lte"]
    },

    "disaster": {
        "breeds": disaster_query["breed"]["$in"],
        "sex": disaster_query["sex_upon_outcome"],
        "minimum_age": disaster_query[
            "age_upon_outcome_in_weeks"
        ]["$gte"],
        "maximum_age": disaster_query[
            "age_upon_outcome_in_weeks"
        ]["$lte"]
    }
}


def calculate_candidate_score(animal, rescue_profile):
    """
    Calculate a rescue candidate score out of 100 points.

    Breed match: 40 points
    Sex match: 25 points
    Age match: 25 points
    Valid coordinates: 5 points
    Known name: 5 points
    """

    score = 0

    animal_breed = str(
        animal.get("breed", "")
    ).lower()

    preferred_breeds = rescue_profile["breeds"]

    breed_match = any(
        preferred_breed.lower() in animal_breed
        for preferred_breed in preferred_breeds
    )

    if breed_match:
        score += 40

    animal_sex = animal.get("sex_upon_outcome")

    if animal_sex == rescue_profile["sex"]:
        score += 25

    animal_age = pd.to_numeric(
        animal.get("age_upon_outcome_in_weeks"),
        errors="coerce"
    )

    if (
        pd.notna(animal_age)
        and rescue_profile["minimum_age"]
        <= animal_age
        <= rescue_profile["maximum_age"]
    ):
        score += 25

    latitude = pd.to_numeric(
        animal.get("location_lat"),
        errors="coerce"
    )

    longitude = pd.to_numeric(
        animal.get("location_long"),
        errors="coerce"
    )

    if pd.notna(latitude) and pd.notna(longitude):
        score += 5

    animal_name = animal.get("name")

    if (
        animal_name is not None
        and not pd.isna(animal_name)
        and str(animal_name).strip()
    ):
        score += 5

    return score


def select_top_candidates(records, rescue_profile, top_k):
    """
    Use a minimum heap to retain the top K rescue candidates.

    Processing N records with a heap of size K has a time
    complexity of O(N log K).
    """

    if not records or top_k <= 0:
        return []

    candidate_heap = []

    for record_number, animal in enumerate(records):

        score = calculate_candidate_score(
            animal,
            rescue_profile
        )

        # The record number acts as a tie breaker when two animals
        # receive the same candidate score.
        candidate_entry = (
            score,
            -record_number,
            animal
        )

        if len(candidate_heap) < top_k:
            heapq.heappush(
                candidate_heap,
                candidate_entry
            )

        elif candidate_entry[:2] > candidate_heap[0][:2]:
            heapq.heapreplace(
                candidate_heap,
                candidate_entry
            )

    # Only the K selected records need to be sorted for display.
    ranked_candidates = sorted(
        candidate_heap,
        key=lambda candidate: (
            candidate[0],
            candidate[1]
        ),
        reverse=True
    )

    results = []

    for rank, candidate in enumerate(
        ranked_candidates,
        start=1
    ):
        score = candidate[0]
        animal = candidate[2]

        ranked_animal = {
            "candidate_rank": rank,
            "candidate_score": score
        }

        ranked_animal.update(animal)
        results.append(ranked_animal)

    return results


@lru_cache(maxsize=12)
def get_ranked_candidates(filter_value, cache_time):
    """
    Retrieve and rank the rescue candidates.

    The cache_time value changes once every 60 seconds, allowing
    recent results to be reused while still refreshing periodically.
    """

    rescue_profile = rescue_profiles.get(filter_value)

    if rescue_profile is None:
        return []

    # Evaluate all dogs so the algorithm can identify the best
    # available candidates instead of requiring an exact match.
    animal_records = shelter.read({
        "animal_type": "Dog"
    })

    return select_top_candidates(
        animal_records,
        rescue_profile,
        TOP_K
    )


# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an
# invalid object type of 'ObjectID' - which will cause the data_table to crash
df.drop(columns=["_id"], inplace=True, errors="ignore")

# Add columns for the algorithm results.
df.insert(0, "candidate_rank", "")
df.insert(1, "candidate_score", "")

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################

app = Dash(__name__)


app.layout = html.Div([
    html.Div(
        id="hidden-div",
        style={"display": "none"}
    ),

    html.Center(
        html.B(
            html.H1(
                "SNHU CS-340 Ricky Suarez Dashboard"
            )
        )
    ),

    html.Center(
        html.Img(
            src=app.get_asset_url(
                "Grazioso_Salvare_Logo.png"
            ),
            style={"width": "300px"}
        )
    ),

    html.Hr(),

    # Added Radio Option rather than buttons
    dcc.RadioItems(
        id="rescue-filter",
        options=[
            {
                "label": "Water Rescue",
                "value": "water"
            },
            {
                "label": "Wilderness Rescue",
                "value": "wilderness"
            },
            {
                "label": "Disaster / Tracking",
                "value": "disaster"
            },
            {
                "label": "Reset",
                "value": "reset"
            }
        ],
        value="reset",
        labelStyle={"display": "block"}
    ),

    dash_table.DataTable(
        id="datatable-id",

        columns=[
            {
                "name": i,
                "id": i,
                "deletable": False,
                "selectable": True
            }
            for i in df.columns
        ],

        data=df.to_dict("records"),

        # Set up the features for your interactive data table.
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable=False,
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        id="map-id",
        className="col s12 m6"
    ),

    # Added secondary Bar Graph
    html.Br(),

    dcc.Graph(
        id="outcome-chart"
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output("datatable-id", "data"),
    Input("rescue-filter", "value")
)
def update_table(filter_value):

    # Reset displays all records without candidate rankings.
    if filter_value == "reset":

        filtered_df = pd.DataFrame.from_records(
            shelter.read(reset_query)
        )

        filtered_df.drop(
            columns=["_id"],
            inplace=True,
            errors="ignore"
        )

        filtered_df.insert(
            0,
            "candidate_rank",
            ""
        )

        filtered_df.insert(
            1,
            "candidate_score",
            ""
        )

        return filtered_df.to_dict("records")

    if filter_value not in rescue_profiles:
        return []

    # Change the cache key once every 60 seconds.
    cache_time = int(time.time() // 60)

    ranked_records = get_ranked_candidates(
        filter_value,
        cache_time
    )

    filtered_df = pd.DataFrame.from_records(
        ranked_records
    )

    filtered_df.drop(
        columns=["_id"],
        inplace=True,
        errors="ignore"
    )

    return filtered_df.to_dict("records")


# Highlight a column when the user selects it.
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns")
)
def update_styles(selected_columns):

    if not selected_columns:
        return []

    return [
        {
            "if": {
                "column_id": column
            },
            "background_color": "#D2F3FF"
        }
        for column in selected_columns
    ]


# Update the map for the selected data entry.
@app.callback(
    Output("map-id", "children"),
    [
        Input(
            "datatable-id",
            "derived_virtual_data"
        ),
        Input(
            "datatable-id",
            "derived_virtual_selected_rows"
        )
    ]
)
def update_map(viewData, index):

    # Prevent an error when the table contains no records.
    if not viewData:
        return html.Div(
            "No data available for map"
        )

    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        return html.Div(
            "No data available for map"
        )

    # Use the first record when no row has been selected.
    if not index:
        row = 0
    else:
        row = index[0]

    # Prevent an outdated selection from accessing an invalid row.
    if row < 0 or row >= len(dff):
        row = 0

    required_columns = [
        "location_lat",
        "location_long",
        "breed",
        "name"
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in dff.columns
    ]

    if missing_columns:
        return html.Div(
            "The map cannot be displayed because required columns "
            "are missing."
        )

    latitude = pd.to_numeric(
        dff.loc[row, "location_lat"],
        errors="coerce"
    )

    longitude = pd.to_numeric(
        dff.loc[row, "location_long"],
        errors="coerce"
    )

    if pd.isna(latitude) or pd.isna(longitude):
        return html.Div(
            "The selected animal does not have valid coordinates."
        )

    animal_breed = dff.loc[row, "breed"]
    animal_name = dff.loc[row, "name"]

    if pd.isna(animal_breed):
        animal_breed = "Unknown breed"

    if pd.isna(animal_name):
        animal_name = "Unknown"

    return [
        dl.Map(
            style={
                "width": "1000px",
                "height": "500px"
            },

            center=[
                float(latitude),
                float(longitude)
            ],

            zoom=10,

            children=[
                dl.TileLayer(
                    id="base-layer-id"
                ),

                dl.Marker(
                    position=[
                        float(latitude),
                        float(longitude)
                    ],

                    children=[
                        dl.Tooltip(
                            str(animal_breed)
                        ),

                        dl.Popup([
                            html.H1(
                                "Animal Name"
                            ),

                            html.P(
                                str(animal_name)
                            )
                        ])
                    ]
                )
            ]
        )
    ]


@app.callback(
    Output("outcome-chart", "figure"),
    Input("datatable-id", "derived_virtual_data")
)
def update_outcome_chart(rows):
    """
    Use a MongoDB aggregation pipeline to count outcome types for
    the records currently displayed in the dashboard table.
    """

    if rows is None or len(rows) == 0:
        return px.bar(
            title="Outcome Type Frequency"
        )

    animal_ids = [
        row.get("animal_id")
        for row in rows
        if row.get("animal_id") is not None
        and str(row.get("animal_id")).strip()
    ]

    if not animal_ids:
        return px.bar(
            title="Outcome Type Frequency"
        )

    outcome_records = shelter.aggregate_outcome_counts(
        animal_ids
    )

    if not outcome_records:
        return px.bar(
            title="Outcome Type Frequency"
        )

    outcome_counts = pd.DataFrame.from_records(
        outcome_records
    )

    fig = px.bar(
        outcome_counts,
        x="Outcome Type",
        y="Count",
        title="Outcome Type Frequency"
    )

    return fig


# Run the dashboard.
app.run(
    port=8056,
    debug=True
)